# Two-reactor methanol synthesis: HYSYS optimization

Minimizes the cost `Z` (€/t) of the two-reactor flowsheet in Aspen HYSYS over six variables (P1, T1, T2, L1, L2, purge). It runs in two phases: a global search with differential evolution, then polishing with Nelder-Mead. At the end it applies the best result to the open HYSYS case.

Requires Windows with Aspen HYSYS installed, and the path to the simulation (`hyFilePath`) the file name set in the first line of the first code cell (the `.hsc` file goes in this folder).

In [ ]:
hyFilePath = "EnergOptTwoReactorSimulation_DENM.hsc"  # HYSYS simulation case (.hsc), placed in this folder

import win32com.client as win32
from scipy.optimize import minimize, differential_evolution
import os
import time
import sys
import numpy as np
import pandas as pd

t0 = time.time()

## Configuration

Optimization variables and bounds (6 variables).

In [ ]:
MAX_WAIT_S    = 10.0
POLL_INTERVAL = 0.05

# GENERAL CONFIGURATION (6 variables)
nombres_vars = ["P1", "T1", "T2", "L1", "L2", "Purge"]
limites = [
    (60.0, 90.0),    # P1 (bar)
    (200.0, 250.0),  # T1 (ºC)
    (200.0, 250.0),  # T2 (ºC)
    (0.1, 15.0),     # L1 (m)
    (0.1, 15.0),     # L2 (m)
    (0.01, 0.15),    # Purge
]
dim = len(limites)

historia_x, historia_z = [], []

## 1. HYSYS connection

In [ ]:
print('--- STARTING HYSYS CONNECTION ---')
# HYSYS needs a full path, so resolve the file name against this notebook's folder
hyFilePath = os.path.abspath(hyFilePath)

try:
    hyApp  = win32.Dispatch('HYSYS.Application')
    hyCase = hyApp.SimulationCases.Open(hyFilePath)
    hyCase.Visible = True

    hy_f      = hyCase.Flowsheet
    hy_solver = hyCase.Solver
    hy_sprd   = hy_f.Operations.Item("OptimizerSpreadsheet")

    stream_P1 = hy_f.MaterialStreams.Item("Feed_2")
    stream_T1 = hy_f.MaterialStreams.Item("Feed_3")
    stream_T2 = hy_f.MaterialStreams.Item("Feed_8")
    reactor_1 = hy_f.Operations.Item("PFR-100")
    reactor_2 = hy_f.Operations.Item("PFR-100-2")

    cell_purge = hy_sprd.Cell("E10")
    cell_Z     = hy_sprd.Cell("E14")

    recycle_op = hy_f.Operations.Item("RCY-1")
    recycle_stream = hy_f.MaterialStreams.Item("Recycle_4")

    print('HYSYS setup completed\n')

except Exception as e:
    print(f"Error connecting to HYSYS: {e}")
    sys.exit()

## 2. Objective function

Sets the variables in HYSYS, waits for the solver, and returns `Z`. Failed evaluations return `1e10`; after 10 consecutive failures it aborts.

In [ ]:
_fail_streak = 0
_FAIL_LIMIT  = 10
_eval_count  = 0

def objetivo(x):
    global _fail_streak, _eval_count
    p1, t1, t2, l1, l2, f_purge = x
    _eval_count += 1

    try:
        hy_solver.CanSolve = False

        stream_P1.Pressure.SetValue(p1, "bar")
        stream_T1.Temperature.SetValue(t1, "C")
        stream_T2.Temperature.SetValue(t2, "C")
        reactor_1.TubeLength.SetValue(l1, "m")
        reactor_2.TubeLength.SetValue(l2, "m")
        cell_purge.CellValue = f_purge

        hy_solver.CanSolve = True

        deadline = time.perf_counter() + MAX_WAIT_S
        while hy_solver.IsSolving:
            if time.perf_counter() > deadline:
                print("  [WARN] Solver timeout — discarding point")
                hy_solver.CanSolve = False
                return 1e10
            time.sleep(POLL_INTERVAL)

        valor_z = cell_Z.CellValue

        if valor_z is None or valor_z <= 0:
            return 1e10

        historia_x.append([p1, t1, t2, l1, l2, f_purge])
        historia_z.append(valor_z)

        _fail_streak = 0
        print(f"  [{_eval_count:03d}] P1:{p1:.1f}  T1:{t1:.1f}  T2:{t2:.1f}  "
              f"L1:{l1:.2f}  L2:{l2:.2f}  Purge:{f_purge:.4f}  =>  Z:{valor_z:.2f}")
        return valor_z

    except Exception as e:
        _fail_streak += 1
        print(f"  [ERROR] Iter failed ({_fail_streak}/{_FAIL_LIMIT}): {e}")
        try:
            hy_solver.CanSolve = False
            recycle_op.IsIgnored = True
            recycle_stream.MolarFlow.SetValue(1, "kgmole/h")
            hy_solver.CanSolve = True
            time.sleep(1)


            hy_solver.CanSolve = False
            recycle_op.IsIgnored = False
            hy_solver.CanSolve = True
            time.sleep(1)
        except Exception:
            pass
        if _fail_streak >= _FAIL_LIMIT:
            print("Too many consecutive failures — aborting.")
            sys.exit(1)
        return 1e10

## 3. Phase 1: global search (differential evolution, tuned)

In [ ]:
# DE — changes with respect to the original, and why:
#   - popsize: 90 -> 15.  popsize is a MULTIPLIER of the dimension, not an
#     absolute number of agents. With dim=6, popsize=90 generated 540 agents
#     per generation (!). popsize=15 gives 90 agents — generous for 6D without
#     being disproportionate. This is the biggest source of computational savings.
#   - init='sobol': more uniform coverage of the space than the default
#     'latinhypercube', for the same number of points.
#   - mutation=(0.5, 1.5): somewhat wider range than the default (0.5, 1.0)
#     to favor larger steps on a smooth landscape (suitable for an
#     equilibrium-limited process, with no strong discontinuities).
#   - recombination=0.8: somewhat higher than the default (0.7) so that
#     agents mix information more aggressively, speeding up
#     convergence in a space with correlated variables (recycle).
#   - maxiter=120: explicit safety ceiling (previously unlimited).
#   - tol=0.01: slightly more permissive than the original (0.005) since
#     Nelder-Mead handles the final refinement.
#   - polish=False: polishing is done by Nelder-Mead afterwards, with its
#     own tolerance control (cheaper than DE's internal "polish",
#     which uses L-BFGS-B and does not handle HYSYS solver noise well).

print("--- PHASE 1: GLOBAL SEARCH (Differential Evolution, tuned) ---")
res_global = differential_evolution(
    objetivo,
    bounds        = limites,
    popsize       = 15,
    init          = 'sobol',
    mutation      = (0.5, 1.5),
    recombination = 0.8,
    tol           = 0.01,
    maxiter       = 120,
    seed          = 42,
    polish        = False,
    disp          = True,
)

print(f"\n  DE best (reported): Z={res_global.fun:.2f}  x={res_global.x}")

## 4. Phase 2: polishing (Nelder-Mead, adaptive)

In [ ]:
# Nelder-Mead — changes with respect to the original, and why:
#   - adaptive=True: SCALES the simplex parameters according to the dimension
#     (recommended by scipy for problems >2D). This matters here
#     because your variables have very different scales (L: 0.1-15 m vs.
#     Purge: 0.01-0.15) — without it, the initial simplex can collapse in
#     one direction and expand poorly in another.
#   - xatol=0.02, fatol=0.1: stricter than the original (0.05, 0.5),
#     since this is now the ONLY polishing phase (no SLSQP after it).
#   - maxiter=150: a bit more margin than the original (100) since it is
#     the final phase responsible for the precision of the result.

print("\n--- PHASE 2: POLISHING (Nelder-Mead, adaptive) ---")
res_final = minimize(
    objetivo,
    res_global.x,
    method  = 'Nelder-Mead',
    bounds  = limites,
    options = {
        'xatol'   : 0.02,
        'fatol'   : 0.1,
        'maxiter' : 150,
        'adaptive': True,
    }
)

print(f"\n  Nelder-Mead result: Z={res_final.fun:.2f}  x={res_final.x}")

## 5. Best result

Scans the whole evaluation history, since the best point is not always the final one.

In [ ]:
t1 = time.time()
print(f"\nExecution time: {t1 - t0:.1f} s")

# ── BEST ACTUAL RESULT: scan the whole history ────────────────────────────────
best_idx = int(np.argmin(historia_z))
best_x   = historia_x[best_idx]
best_z   = historia_z[best_idx]

print("\n" + "="*60)
print("BEST ACTUAL RESULT (across all iterations)")
print("="*60)
for nombre, valor in zip(nombres_vars, best_x):
    if nombre == "Purge":
        print(f"   -> {nombre}: {valor*100:.3f} %")
    else:
        print(f"   -> {nombre}: {valor:.2f}")
print(f"   -> Cost Z:       {best_z:.2f} €/t")
print(f"   -> Evaluations:{len(historia_z)}")
print("="*60)

## 6. Apply the best result to the HYSYS simulation

In [ ]:
print("\n--- APPLYING BEST RESULT TO THE HYSYS SIMULATION ---")
try:
    p1_best, t1_best, t2_best, l1_best, l2_best, purge_best = best_x

    hy_solver.CanSolve = False
    stream_P1.Pressure.SetValue(p1_best, "bar")
    stream_T1.Temperature.SetValue(t1_best, "C")
    stream_T2.Temperature.SetValue(t2_best, "C")
    reactor_1.TubeLength.SetValue(l1_best, "m")
    reactor_2.TubeLength.SetValue(l2_best, "m")
    cell_purge.CellValue = purge_best
    hy_solver.CanSolve = True

    deadline = time.perf_counter() + MAX_WAIT_S
    while hy_solver.IsSolving:
        if time.perf_counter() > deadline:
            print("  [WARN] Solver timeout while applying final result.")
            break
        time.sleep(POLL_INTERVAL)

    z_check = cell_Z.CellValue
    print(f"  Simulation updated. Z verified in HYSYS: {z_check:.2f} €/t")

    if abs(z_check - best_z) > 1.0:
        print(f"  [WARNING] Verified Z ({z_check:.2f}) differs from the history ({best_z:.2f}) "
              f"by more than 1.0 — possible non-convergence or solver noise.")

except Exception as e:
    print(f"  [ERROR] Could not apply the final result to HYSYS: {e}")